# 02_clean_prices.py 결과 확인

`data/cleaned/prices`의 결측 보간(`is_interpolated`) / 액면분할 의심 탐지(`split_suspected`) 결과를 확인한다.

## 정제 대상 연도

`jobs/02_clean_prices.py`는 `--years`로 정제 대상 연도를 좁힐 수 있다(미지정 시 `data/raw/prices` 전체).

백테스트 대상 연도(2021~2023) 외 연도가 섞이면 거래일 캘린더가 실제보다 넓어져 지표가 왜곡된다. 이 리포에서는 04번 모멘텀(t-12개월) 계산에 필요한 2020년까지 포함해 `--years 2020,2021,2022,2023`으로 정제했다.

## "보간된 행 0건"이 나오는 이유 (실측, 2026-08-04)

`01_ingest_raw.py`의 `fetch_krx_listed`는 `--bas-dt`(이 데이터의 companies 파티션 기준 `20260724`, 즉 2026년 시점) 기준 상장종목 목록으로 가격을 수집한다.

이 목록에는 2020~2023년 당시엔 아직 상장 전이었던 종목도 포함된다. 그런 종목의 2020~2023년 구간은 `data/raw/prices`에 시세 자체가 없다.

`fill_missing_trading_days()`의 forward-fill은 "종목이 처음 등장한 날짜 이전" 구간을 채울 직전값이 없어 그대로 결측(null)으로 남고, `main()`의 107~110행이 이런 행(보간 불가)을 걸러낸다.

실측 확인 결과:
- 이번 정제에서 나온 결측 121,564건은 전부 "그 종목이 raw에 최초로 등장한 날짜 이전" 구간이었다(`bas_dt < first_seen`).
- 그 이후 구간의 결측(진짜 거래정지로 추정되는 케이스)은 0건이었다.

즉 이 데이터셋에는 2021~2023년 구간 안에서 실제 거래정지로 인한 결측이 없다는 뜻이며, 아래 "1. 보간된 행"이 0건인 것은 버그가 아니라 이 실측 결과를 정확히 반영한 값이다.</cell id="552db7f1">


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.master("spark://spark-master:7077").appName("check_cleaned_prices").getOrCreate()

# 02_clean_prices.py --years와 동일 범위(2020~2023). is_interpolated/split_suspected는 02번
# 실행 시점에 이미 이 범위로 계산되어 저장돼 있으므로, 아래 필터는 결과값을 바꾸지 않고
# cleaned/prices에 남아있는 다른 연도 파티션(2025/2026, 이전 재실행 잔여분)만 제외한다.
YEARS = ["2020", "2021", "2022", "2023"]

df_all = spark.read.parquet("/opt/spark-apps/data/cleaned/prices")
print(f"전체(전 연도) {df_all.count()}건")

df = df_all.filter(F.col("year").isin(YEARS))
print(f"백테스트 대상 연도({','.join(YEARS)}) {df.count()}건")
df.groupBy("snapshot_type").count().toPandas()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/08/04 10:41:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


전체(전 연도) 1806064건


백테스트 대상 연도(2020,2021,2022,2023) 1765334건


,snapshot_type,count
0,current,1752091
1,12m_ago,6496
2,1m_ago,6747


## 1. 보간된 행 (`is_interpolated=true`)
직전 종가로 채운 결측 거래일. 거래정지 구간도 원본에 시세가 없는 형태로 들어오므로 여기 포함된다.

In [2]:
interpolated = df.filter(F.col("is_interpolated"))
print(f"보간된 행: {interpolated.count()}건")
interpolated.select("stock_code", "bas_dt", "close_price", "is_interpolated") \
    .orderBy("stock_code", "bas_dt") \
    .toPandas()

보간된 행: 0건


,stock_code,bas_dt,close_price,is_interpolated


## 2. 액면분할/병합 의심 행 (`split_suspected=true`)
전일 대비 종가 등락률이 -40% 이하 또는 +67% 이상인 지점. 자동 보정 없이 플래그만 표시됨 — 진짜 분할/병합인지, 단순 급등락인지는 별도 확인 필요.

In [3]:
w = Window.partitionBy("stock_code").orderBy("bas_dt")
suspects = df.withColumn("prev_close", F.lag("close_price").over(w)) \
    .filter(F.col("split_suspected")) \
    .withColumn("day_over_day_rate", F.round((F.col("close_price") - F.col("prev_close")) / F.col("prev_close") * 100, 2))

print(f"액면분할/병합 의심 행: {suspects.count()}건")
suspects.select("stock_code", "bas_dt", "prev_close", "close_price", "day_over_day_rate") \
    .orderBy("stock_code", "bas_dt") \
    .toPandas()

액면분할/병합 의심 행: 1037건


,stock_code,bas_dt,prev_close,close_price,day_over_day_rate
0,000040,20200224,242,656,171.07
1,000100,20200423,229000,47600,-79.21
2,000150,20200324,55800,32600,-41.58
3,000220,20210330,14700,7460,-49.25
4,000370,20200324,2205,1215,-44.90
...,...,...,...,...,...
1032,950130,20200824,8810,29150,230.87
1033,950140,20200324,4600,2515,-45.33
1034,950160,20221025,8010,20850,160.30
1035,950170,20200324,6000,3170,-47.17


## 3. 정제 후에도 남은 결측치 (컬럼별 null 개수)
전부 0이어야 정상.

In [4]:
null_counts = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns])
null_counts.toPandas()

,stock_code,bas_dt,close_price,fluctuation_rate,listed_share_count,market_cap,open_price,snapshot_type,is_interpolated,split_suspected,year
0,0,0,0,0,0,0,0,0,0,0,0


In [5]:
spark.stop()